<a href="https://colab.research.google.com/github/Andreas-Lukito/Stock_Sentiment_Analysis/blob/dev%2Fandreas/notebooks/04_FinBERT_categorical.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FinBERT for Predicting News Sentiment

## Install Libraries

In [ ]:
# ! pip install contractions emoji gensim optuna torch matplotlib

## Iport Libraries

In [ ]:
# Common Python Libraries
import numpy as np
import pandas as pd
import os
import sys
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import random

# Deep Learning Libraries
import torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import Adam

# Data Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder

## Download nltk dependencies
import nltk
nltk.download('stopwords')
nltk.download('punkt_tab')

# Model metrics
from sklearn.metrics import classification_report, confusion_matrix

# Google Colab Setup
# from google.colab import drive
# drive.mount('/content/drive')
# project_path = "/content/drive/MyDrive/stock_news_sentiment_analysis"

project_path = "../"

# Add the path to the text preprocessor
sys.path.append(os.path.abspath(os.path.join(project_path, "lib")))

## Import preprocessor
from preprocessor import clean_text

# Project Seed for Reproducability
SEED = random.randint(0, 2**32 - 1)  # Random integer between 0 and 2^32-1
print(f"seed: {SEED}")

model_name = "ProsusAI/finbert"

/home/horizontal_roterien_katze/miniforge3/envs/thesis/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


seed: 709430354


[nltk_data] Downloading package stopwords to
[nltk_data]     /home/horizontal_roterien_katze/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/horizontal_roterien_katze/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## Choose Device

In [ ]:
# Detect available device
if torch.cuda.is_available():
    # check if ROCm backend is active
    if torch.version.hip is not None:
        backend = "ROCm"
    else:
        backend = "CUDA"

    device = torch.device("cuda")
    print(f"PyTorch is using GPU: {torch.cuda.get_device_name(0)}")
    print(f"Backend: {backend}")
else:
    device = torch.device("cpu")
    print("PyTorch is not using GPU — running on CPU")

PyTorch is using GPU: AMD Radeon 880M
Backend: ROCm


/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory


## Import Data

In [ ]:
before_date = "2025-11"

# Data path
categorized_data_path = os.path.join(project_path,f"news_cache/catgorized_data/categorized_news_data.csv")

# Import Data
news_data = pd.read_csv(filepath_or_buffer=categorized_data_path, sep=',')

In [ ]:
news_data.head()

,index,uuid,title,description,keywords,snippet,url,image_url,language,published_at,source,relevance_score,entities,similar,sentiment,text,length,clean_text,categorical_sentiment_3_class
0,0,487e6a88-d3c2-4ae1-8dc2-26af6b31d688,2025: The Year Of Alphabet (GOOG),No stock has seen a bigger jump recently than ...,NaN,vzphotos/iStock Editorial via Getty Images\n\n...,https://seekingalpha.com/article/4848680-2025-...,https://static.seekingalpha.com/cdn/s3/uploads...,en,2025-11-30T05:30:00.000000Z,seekingalpha.com,NaN,"[{'symbol': 'GOOGL', 'name': 'Alphabet Inc.', ...",[],0.0000,vzphotos/iStock Editorial via Getty Images\n\n...,42,vzphotos istock editorial via getty images sin...,neutral
1,1,92b5c2bd-d324-4ae8-b115-2cfd95a8fa98,Why I'm Doubling Down On My Adobe Position (NA...,"Adobe's revenue is highly predictable, driven ...",NaN,To say that Adobe ( ADBE ) stock has not had a...,https://seekingalpha.com/article/4848762-why-i...,https://static.seekingalpha.com/cdn/s3/uploads...,en,2025-11-30T05:25:01.000000Z,seekingalpha.com,NaN,"[{'symbol': 'ADBE', 'name': 'Adobe Inc.', 'exc...",[],0.0000,To say that Adobe ( ADBE ) stock has not had a...,259,to say that adobe adbe stock has not had a goo...,neutral
2,2,9084e5f1-75f5-4f15-aa3d-0676073b4aaf,Global week ahead: The start of a Santa Rally ...,NaN,"STOXX 600, business news",And just like that... December is upon us. It'...,https://www.cnbc.com/2025/11/30/global-week-ah...,https://image.cnbcfm.com/api/v1/image/10823257...,en,2025-11-30T05:10:58.000000Z,cnbc.com,NaN,"[{'symbol': 'M', 'name': ""Macy's, Inc."", 'exch...",[],0.6908,And just like that... December is upon us. It'...,493,and just like that december is upon us it is b...,positive
3,3,487e6a88-d3c2-4ae1-8dc2-26af6b31d688,2025: The Year Of Alphabet (GOOG),No stock has seen a bigger jump recently than ...,NaN,vzphotos/iStock Editorial via Getty Images\n\n...,https://seekingalpha.com/article/4848680-2025-...,https://static.seekingalpha.com/cdn/s3/uploads...,en,2025-11-30T05:30:00.000000Z,seekingalpha.com,NaN,"[{'symbol': 'GOOGL', 'name': 'Alphabet Inc.', ...",[],0.0000,vzphotos/iStock Editorial via Getty Images\n\n...,42,vzphotos istock editorial via getty images sin...,neutral
4,4,92b5c2bd-d324-4ae8-b115-2cfd95a8fa98,Why I'm Doubling Down On My Adobe Position (NA...,"Adobe's revenue is highly predictable, driven ...",NaN,To say that Adobe ( ADBE ) stock has not had a...,https://seekingalpha.com/article/4848762-why-i...,https://static.seekingalpha.com/cdn/s3/uploads...,en,2025-11-30T05:25:01.000000Z,seekingalpha.com,NaN,"[{'symbol': 'ADBE', 'name': 'Adobe Inc.', 'exc...",[],0.0000,To say that Adobe ( ADBE ) stock has not had a...,259,to say that adobe adbe stock has not had a goo...,neutral


In [ ]:
news_data.isna().sum()

index                                0
uuid                                 0
title                                0
description                       3148
keywords                         37795
snippet                            216
url                                  0
image_url                          389
language                             0
published_at                         0
source                               0
relevance_score                  77088
entities                             0
similar                              0
sentiment                            4
text                             24181
length                               0
clean_text                       24181
categorical_sentiment_3_class        0
dtype: int64

In [ ]:
news_data = news_data.dropna(subset=["clean_text"])

In [ ]:
news_data.isna().sum()

index                                0
uuid                                 0
title                                0
description                       2915
keywords                         23050
snippet                             75
url                                  0
image_url                          255
language                             0
published_at                         0
source                               0
relevance_score                  52907
entities                             0
similar                              0
sentiment                            3
text                                 0
length                               0
clean_text                           0
categorical_sentiment_3_class        0
dtype: int64

In [ ]:
value_maps = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

news_data["categorical_sentiment_3_class_value_maps"] = news_data["categorical_sentiment_3_class"].map(value_maps)

## Split the data to Train, Test, and Validation

In [ ]:
test_size = 0.20
val_size = 0.10

# Splitting the data into train and temp (which will be further split into validation and test)
train_df, test_df = train_test_split(news_data, test_size=test_size, random_state=SEED, stratify=news_data["categorical_sentiment_3_class_value_maps"])

# Splitting train into validation and test sets
val_df, test_df = train_test_split(test_df, test_size=val_size, random_state=SEED, stratify=test_df["categorical_sentiment_3_class_value_maps"])

In [ ]:
x_train = train_df["clean_text"].tolist()
y_train = train_df["categorical_sentiment_3_class_value_maps"].tolist()

x_test = test_df["clean_text"].tolist()
y_test = test_df["categorical_sentiment_3_class_value_maps"].tolist()

x_val = val_df["clean_text"].tolist()
y_val = val_df["categorical_sentiment_3_class_value_maps"].tolist()

## Data Preprocessing

### Tokenizer for the text

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

class sentiment_text(torch.utils.data.Dataset): # create a class that behaves like torch.utils.data.Dataset
    def __init__(self, texts, labels, tokenizer):
        self.encodings = tokenizer( # converts raw text -> model input
                                    texts,
                                    truncation = True,
                                    padding = True,
                                    max_length = 300 # since the max length of the tweets are around 35 - 40 words
                                )

        # get the labels
        self.labels = labels

    def __getitem__(self, index): # so that pytorch can get the data (returns one sample of the data)
        item = {key: torch.tensor(val[index]) for key, val in self.encodings.items()} # self.encoding stores (input_ids, attention_mask, label)
        item["labels"] = torch.tensor(self.labels[index], dtype=torch.float32) # get the label on the chosen index while converting to a torch tensor format
        return item

    def __len__(self): #to get the length of the data (used when batching)
        return len(self.labels)

In [ ]:
# Create the dataset for training, testing and validation
train_dataset = sentiment_text(x_train, y_train, tokenizer)
test_dataset  = sentiment_text(x_test, y_test, tokenizer)
val_dataset  = sentiment_text(x_val, y_val, tokenizer)

### Data Loader for the Model

In [ ]:

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=32)
val_loader  = DataLoader(val_dataset, batch_size=32)

## Train Model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3, # Since there are three classes ["Negative", "Neutral", "Positive"]
    ignore_mismatched_sizes=True
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 44375.99it/s]
BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
optimizer = Adam(model.parameters(), lr=5e-5)

In [71]:
model.to(device)
model.train() #make the model to training mode

for epoch in tqdm(range(10), desc="Training FinBERT Model", unit="epoch"):  # number of epochs

    for batch in train_loader:
        for k, v in batch.items():
            batch[k] = v.to(device)

        optimizer.zero_grad() # Resets all gradients to zero before computing new ones.
        outputs = model(**batch)  # forward pass
        loss = outputs.loss
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1} loss: {loss.item()}")

Training FinBERT Model:   1%|          | 1/100 [07:56<13:06:37, 476.74s/epoch]

Epoch 1 | Train Loss: 1.2276 | Val Loss: 1.1168


Training FinBERT Model:   2%|▏         | 2/100 [15:53<12:59:03, 476.98s/epoch]

Epoch 2 | Train Loss: 1.2329 | Val Loss: 1.2000


Training FinBERT Model:   3%|▎         | 3/100 [23:50<12:51:11, 477.02s/epoch]

...

Epoch 14 | Train Loss: 1.2401 | Val Loss: 1.1849


Training FinBERT Model:  15%|█▌        | 15/100 [1:59:15<11:15:40, 476.95s/epoch]

Epoch 15 | Train Loss: 1.2216 | Val Loss: 1.2074


Training FinBERT Model:  15%|█▌        | 15/100 [2:07:12<12:00:48, 508.80s/epoch]

Epoch 16 | Train Loss: 1.2348 | Val Loss: 1.2320
Early stopping triggered


## Model Evaluation

In [72]:
def evaluate_model(model, data_loader, device):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in data_loader:
            # move batch to device
            for k, v in batch.items():
                batch[k] = v.to(device)

            # forward pass
            outputs = model(**batch) # turns raw inputs to named inputs as hugging face expects

            preds = outputs.logits.squeeze(-1)
            labels = batch["labels"].squeeze(-1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # metrics
    classification_report = classification_report(all_labels, all_preds, target_names=["Negative", "Neutral", "Positive"], zero_division=0)
    confusion_matrix = confusion_matrix(all_labels, all_preds, target_names=["Negative", "Neutral", "Positive"])

    return classification_report, confusion_matrix


In [73]:
classification_report, confusion_matrix = evaluate_model(
                                        model,
                                        test_loader,
                                        device
                                        )

print("========== Classification Report ==========")
print(classification_report)

print("========== Confusion Matrix ==========")
print(confusion_matrix)

========== Classification Report ==========
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       330
           1       0.00      0.00      0.00       270
           2       0.43      1.00      0.60       459

    accuracy                           0.43      1059
   macro avg       0.14      0.33      0.20      1059
weighted avg       0.19      0.43      0.26      1059

========== Confusion Matrix ==========
[[  0   0 330]
 [  0   0 270]
 [  0   0 459]]
